Run a simulation and sample genomes from the transmission network.

Sample at say 1%, 5%, and 10% of the transmissions (to begin with) and reconstruct the phylogeny.

In [13]:
import numpy as np
import matplotlib.pyplot as plt

import covasim as cv
from Bio import SeqIO

from covasim.jhk_covasim4wastewater.intervention_search.simulation.runner import run_evo_simulation
from covasim.jhk_covasim4wastewater.intervention_search.analysis.mutation_tree import build_mutation_tree, build_infection_seq_tree, mutation_tree_to_newick, extract_sequenced_subtree, export_fasta_from_G, infer_tree_jc69

from covasim.transmission_tree import TransmissionTree
from covasim.metrics import Metrics
from covasim.inference import Inference

In [14]:
REF_FASTA     = 'sars-2-data/NC_045512_Hu-1.fasta'
FITNESS_CSV   = '../../data/nt_fitness.csv'
ARTIC_PRIMERS = 'sars-2-data/ARTIC_V4-1.bed'

ay4_fasta   = 'sars-2-data/variants/AY.4_consensus.fasta'
ba2_fasta   = 'sars-2-data/variants/BA.2.9_consensus.fasta'
xbb15_fasta = 'sars-2-data/variants/XBB.1.5.18_consensus.fasta'


In [15]:
ay4 = cv.variant(
    variant        = {},
    label          = 'AY.4',
    days           = 30,
    n_imports      = 60,
    founding_fasta = ay4_fasta,
)
ba2 = cv.variant(
    variant        = {},
    label          = 'BA.2',
    days           = 70,
    n_imports      = 60,
    founding_fasta = ba2_fasta,
)
xbb15 = cv.variant(
    variant        = {},
    label          = 'XBB.1.5',
    days           = 110,
    n_imports      = 60,
    founding_fasta = xbb15_fasta,
)

sample_days = list(range(90, 181, 10))
# Routine clinical: 50 symptomatic agents per day, every 10 days.
cs_routine = cv.ClinicalSequencer(
    days      = sample_days,
    n_samples = 50,
    pool      = 'symptomatic',
    label     = 'CS_routine',
    # sequencing_batch_size defaults to 1, causing immediate release
)

In [16]:
evo_pars = dict(
    enable            = True,
    reference         = REF_FASTA,
    mol_clock_rate    = 1e-3, # ~1 substitution/genome/month at 30 000 nt
    sub_model         = 'JC', # decides which mutations happen
    fitness_model = "bloom_nt", # controls transmission rate
    fitness_data_path = FITNESS_CSV,
)
sim_pars = dict(
    pop_size=5000,
    pop_infected=1,
    start_day='2022-09-01',
    end_day='2023-03-31',
    rand_seed=42,
    verbose=0,    
    variants=[xbb15],
    analyzers=[cs_routine],    
)
policy_pars = dict(
    p_test=0.2,
    p_seq=0.5,
    detection_threshold=3,
    delay_pmf=[0.05, 0.15, 0.30, 0.25, 0.15, 0.07, 0.03],
    quiet_period=0,
)
sim, obs, interv, _, results = run_evo_simulation(sim_pars, evo_pars, policy_pars)
G_infect, mutation_nodes, node_events = build_infection_seq_tree(sim, daily_sequenced_agents=obs.daily_sequenced_agents)
print(f"Policy B mutation tree: nodes={len(G_infect.nodes)}, edges={len(G_infect.edges)}")

cs_routine = sim.get_analyzer('CS_routine')

Policy B mutation tree: nodes=879, edges=878


/home/yutianc/covasim4wastewater/covasim/clinical.py:194: UserWarning: ClinicalSequencer day 140: requested 50 samples but only 43 agents are available in pool='symptomatic'. Returning all 43 agents.
  warnings.warn(
/home/yutianc/covasim4wastewater/covasim/clinical.py:194: UserWarning: ClinicalSequencer day 150: requested 50 samples but only 15 agents are available in pool='symptomatic'. Returning all 15 agents.
  warnings.warn(
/home/yutianc/covasim4wastewater/covasim/clinical.py:194: UserWarning: ClinicalSequencer day 160: requested 50 samples but only 2 agents are available in pool='symptomatic'. Returning all 2 agents.
  warnings.warn(


In [17]:
print(f"pop_size (actual agents):     {sim['pop_size']}")
print(f"pop_scale:                    {sim['pop_scale']}")
print(f"scaled_pop_size:              {sim.scaled_pop_size}")
print(f"rescale:                      {sim['rescale']}")

# current scale factor at end of sim
print(f"final scale factor:           {sim.rescale_vec[-1]:.1f}")

pop_size (actual agents):     5000
pop_scale:                    1
scaled_pop_size:              5000
rescale:                      True
final scale factor:           1.0


In [18]:
log    = sim.people.infection_log
n_muts = [e['n_mutations'] for e in log if 'n_mutations' in e]

print(f'Transmission events:    {len(log):,}')
print(f'Mean branch mutations:  {np.mean(n_muts):.3f}')
print(f'Max branch mutations:   {max(n_muts)}')

Transmission events:    939
Mean branch mutations:  179.481
Max branch mutations:   1411


In [19]:
focus_day = 120
fasta_str = cs_routine.to_fasta(120)
fasta_out = f'./res/clinical_day{focus_day}_0.01.fasta'
with open(fasta_out, 'w') as fh:
    fh.write(fasta_str + '\n')
print(f'Saved clinical FASTA -> {fasta_out}')

Saved clinical FASTA -> ./res/clinical_day120_0.01.fasta


In [20]:
iqtree_inference = Inference(
    fasta_path=f'./res/clinical_day{focus_day}_0.01.fasta',
    prefix="sample_0.01", 
    model="JC",
)
iqtree_inference.run_iqtree_asr()

Running: /home/yutianc/miniforge3/envs/covasim/bin/iqtree -s ./res/clinical_day120_0.01.fasta -m JC -asr -redo -pre sample_0.01
IQ-TREE version 3.1.3 for Linux x86 64-bit built Jul 26 2026
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    profchaos.scripps.edu (AVX512, FMA3, 754 GB RAM)
Command: /home/yutianc/miniforge3/envs/covasim/bin/iqtree -s ./res/clinical_day120_0.01.fasta -m JC -asr -redo -pre sample_0.01
Seed:    714509 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Thu Sep 17 15:46:25 2026
Kernel:  AVX+FMA - 1 threads (384 CPU cores detected)

HINT: Use -nt option to specify number of threads because your CPU has 384 cores!
HINT: -nt AUTO will automatically determine the best number of threads to use.

Reading alignment file ./res/clinical_day120_0.01.fasta ... Fasta format detected
Reading fasta file: 

In [21]:
total_mut, per_branch_mut = iqtree_inference.count_mutations()
print("n mut in the sampled transmission tree:", total_mut)

n mut in the sampled transmission tree: 5427


In [44]:
import pandas as pd

def convert_haplotype_tree_to_long_df(mutation_nodes):
    """
    mutation_nodes: {frozenset of (site, ref, alt) -> node_id}
    Returns long-format df with columns: node, site, ref, alt (numeric).
    """
    rows = []
    for mutation_state, node in mutation_nodes.items():
        if not mutation_state:
            rows.append({"node": node, "site": None, "ref": None, "alt": None})
        for site, ref, alt in mutation_state:
            rows.append({"node": node, "site": int(site), "ref": cv.decode_sequence([ref]), "alt": cv.decode_sequence([alt])})
    return pd.DataFrame(rows)

df_mut_full_tree = convert_haplotype_tree_to_long_df(mutation_nodes)
df_mut_full_tree

,node,site,ref,alt
0,Node_0,NaN,None,None
1,Node_1,160.0,A,T
2,Node_1,18906.0,G,A
3,Node_1,3716.0,C,G
4,Node_1,8962.0,T,C
...,...,...,...,...
168529,Node_878,3699.0,T,C
168530,Node_878,6353.0,C,G
168531,Node_878,14498.0,T,A
168532,Node_878,43.0,C,T


In [23]:
def sanity_check(tree, node_seqs):
    tree_names = {n.name for n in tree.traverse() if n.name}
    state_names = set(node_seqs.keys())
    only_in_tree = tree_names - state_names
    only_in_state = state_names - tree_names
    print(f"Nodes in tree:  {len(tree_names)}")
    print(f"Nodes in state: {len(state_names)}")
    if only_in_tree:
        print(f"  In tree but not .state ({len(only_in_tree)}): {sorted(only_in_tree)[:10]}")
    if only_in_state:
        print(f"  In .state but not tree ({len(only_in_state)}): {sorted(only_in_state)[:10]}")
    if not only_in_tree and not only_in_state:
        print("  All node names match. \u2713")

In [26]:
from ete3 import Tree
from collections import defaultdict

def load_state_file(state_path):
    node_seqs = defaultdict(dict)
    with open(state_path) as f:
        header = None
        for line in f:
            line = line.rstrip("\n")
            if not line or line.startswith("#"):
                continue
            fields = line.split("\t")
            if header is None:
                header = fields
                continue
            row = dict(zip(header, fields))
            node_seqs[row["Node"]][int(row["Site"])] = row["State"]
    return node_seqs

def load_tip_sequences(fasta_path):
    from Bio import SeqIO
    tip_seqs = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        tip_seqs[record.id] = {i + 1: base for i, base in enumerate(str(record.seq))}
    return tip_seqs

iqtree = Tree("/home/yutianc/covasim4wastewater/docs/tutorials/sample_0.01.treefile", format=1)
nodes = load_state_file("/home/yutianc/covasim4wastewater/docs/tutorials/sample_0.01.state") # every site for every node
tips = load_tip_sequences("/home/yutianc/covasim4wastewater/docs/tutorials/res/clinical_day120_0.01.fasta")
nodes.update(tips)
sanity_check(iqtree, nodes)

Nodes in tree:  98
Nodes in state: 98
  All node names match. ✓


In [28]:
def get_mutations(parent_seq, child_seq):
    muts = []
    for site, ref_state in parent_seq.items():
        alt_state = child_seq.get(site)
        if alt_state is not None and alt_state != ref_state:
            muts.append((site, ref_state, alt_state))
    return sorted(muts)

def compute_edge_mutations(tree, node_seqs):
    edge_mutations = {}
    for node in tree.traverse("preorder"): # root to tips
        if node.is_root() or not node.name:
            continue
        parent = node.up
        if parent.name in node_seqs and node.name in node_seqs:
            edge_mutations[node.name] = get_mutations(
                node_seqs[parent.name], node_seqs[node.name] # get the variants
            )
        else:
            edge_mutations[node.name] = []
    return edge_mutations

def compute_cumulative_mutations(tree, edge_mutations):
    cumulative = {}
 
    def recurse(node, parent_cumulative):
        current = frozenset() if node.is_root() else parent_cumulative | set(
            edge_mutations.get(node.name, [])
        )
        if node.name:
            cumulative[node.name] = frozenset(current)
        for child in node.children:
            recurse(child, current)
 
    recurse(tree, frozenset())
    return cumulative

In [45]:
def convert_sample_tree_to_long_df(mut_dict):
    rows = []
    for node, muts in mut_dict.items():
        if not muts:
            rows.append({"node": node, "site": None, "ref": None, "alt": None})
        for site, ref, alt in muts:
            rows.append({"node": node, "site": int(site), "ref": ref, "alt": alt})

    df = pd.DataFrame(rows)
    return df

In [46]:
edge_mut = compute_edge_mutations(iqtree, nodes)
cumultative_mut = compute_cumulative_mutations(iqtree, edge_mut)

df_mut_sampled_tree = convert_sample_tree_to_long_df(cumultative_mut)
df_mut_sampled_tree

,node,site,ref,alt
0,Node1,NaN,None,None
1,agent_450,3132.0,A,T
2,agent_450,15748.0,T,C
3,agent_450,641.0,A,C
4,agent_450,17537.0,C,G
...,...,...,...,...
5665,agent_2918,18181.0,G,C
5666,agent_2918,25524.0,A,G
5667,agent_2918,3962.0,A,G
5668,agent_2918,21099.0,G,T


In [49]:
df_mut_full_tree = df_mut_full_tree.drop_duplicates(subset=["site", "ref", "alt"]) # node is the node id in the graph, each representing a halplotype
df_mut_sampled_tree = df_mut_sampled_tree.drop_duplicates(subset=["site", "ref", "alt"]) # node is the internal node constructed by iqtree inference

common = pd.merge(df_mut_full_tree, df_mut_sampled_tree, how="inner", on=["site", "ref", "alt"])
common

,node_x,site,ref,alt,node_y
0,Node_0,NaN,None,None,Node1
1,Node_1,24396.0,T,G,agent_4812
2,Node_1,2249.0,G,A,agent_21
3,Node_2,2323.0,G,C,agent_2378
4,Node_4,18960.0,G,C,agent_3084
...,...,...,...,...,...
1222,Node_876,20672.0,G,T,agent_117
1223,Node_876,6887.0,A,T,agent_3404
1224,Node_877,29569.0,A,C,agent_256
1225,Node_878,8929.0,T,A,agent_208


In [50]:
print(len(common)/len(df_mut_full_tree))

0.01592741150356322


In [ ]:
# sample rate 5%

cs_routine = cv.ClinicalSequencer(
    days      = sample_days,
    n_samples = 250,
    pool      = 'symptomatic',
    label     = 'CS_routine',
    # sequencing_batch_size defaults to 1, causing immediate release
)

sim = cv.Sim(dict(
    pop_size     = 5_000,
    pop_infected = 1,
        start_day    = '2022-09-01',
        end_day      = '2023-03-31',
    evo_pars = dict(
        enable            = True,
        reference         = REF_FASTA,
        mol_clock_rate    = 1e-6,   # ~1 substitution/genome/month at 30 000 nt
        sub_model         = 'JC', # decides which mutations happen
        fitness_model = "bloom_nt", # controls transmission rate
        fitness_data_path = FITNESS_CSV,
    ),
    variants  = [ay4, ba2, xbb15],
    analyzers = [cs_routine],
    verbose   = 0,
))
sim.run()

cs_routine = sim.get_analyzer('CS_routine')

focus_day = 180 # this has to be the sample day
cs_batch = cs_routine.samples.get(focus_day, [])

print(f'Day {focus_day} ({sim.date(focus_day)}) — clinical estimate vs true simulation proportions:')
print(f'  {"Variant":12s}  {"Clinical (n=50)":>16}  {"True (sim)":>12}  {"Diff":>8}')
print('  ' + '-' * 56)

for v in variant_names:
    clin_pct = 0.0
    if cs_batch:
        counts = variant_counts(cs_batch)
        clin_pct = counts.get(v, 0) / len(cs_batch)
    true_pct = props[v][focus_day]
    diff     = clin_pct - true_pct
    sign     = '+' if diff >= 0 else ''
    print(f'  {v:12s}  {clin_pct:>16.1%}  {true_pct:>12.1%}  {sign}{diff:>7.1%}')

fasta_str = cs_routine.to_fasta(focus_day)
fasta_out = f'./res/clinical_day{focus_day}_0.05.fasta'
with open(fasta_out, 'w') as fh:
    fh.write(fasta_str + '\n')

iqtree_inference = Inference(
    fasta_path=f'./res/clinical_day180_0.05.fasta',
    prefix="sample_0.05", 
    model="JC",
)
iqtree_inference.run_iqtree_asr()

total_mut, per_branch_mut = iqtree_inference.count_mutations()
print("n mut in the sampled transmission tree:", total_mut)

In [ ]:
# sample rate 10%
cs_routine = cv.ClinicalSequencer(
    days      = sample_days,
    n_samples = 500,
    pool      = 'symptomatic',
    label     = 'CS_routine',
    # sequencing_batch_size defaults to 1, causing immediate release
)

sim = cv.Sim(dict(
    pop_size     = 5_000,
    pop_infected = 1,
        start_day    = '2022-09-01',
        end_day      = '2023-03-31',
    evo_pars = dict(
        enable            = True,
        reference         = REF_FASTA,
        mol_clock_rate    = 1e-6,   # ~1 substitution/genome/month at 30 000 nt
        sub_model         = 'JC', # decides which mutations happen
        fitness_model = "bloom_nt", # controls transmission rate
        fitness_data_path = FITNESS_CSV,
    ),
    variants  = [ay4, ba2, xbb15],
    analyzers = [cs_routine],
    verbose   = 0,
))
sim.run()

cs_routine = sim.get_analyzer('CS_routine')

focus_day = 180 # this has to be the sample day
cs_batch = cs_routine.samples.get(focus_day, [])

print(f'Day {focus_day} ({sim.date(focus_day)}) — clinical estimate vs true simulation proportions:')
print(f'  {"Variant":12s}  {"Clinical (n=50)":>16}  {"True (sim)":>12}  {"Diff":>8}')
print('  ' + '-' * 56)

for v in variant_names:
    clin_pct = 0.0
    if cs_batch:
        counts = variant_counts(cs_batch)
        clin_pct = counts.get(v, 0) / len(cs_batch)
    true_pct = props[v][focus_day]
    diff     = clin_pct - true_pct
    sign     = '+' if diff >= 0 else ''
    print(f'  {v:12s}  {clin_pct:>16.1%}  {true_pct:>12.1%}  {sign}{diff:>7.1%}')

fasta_str = cs_routine.to_fasta(focus_day)
fasta_out = f'./res/clinical_day{focus_day}_0.1.fasta'
with open(fasta_out, 'w') as fh:
    fh.write(fasta_str + '\n')
print(f'Saved clinical FASTA -> {fasta_out}')
print(f'  ({len(cs_batch)} sequences, {len(fasta_str)} characters)')

iqtree_inference = Inference(
    fasta_path=f'./res/clinical_day180_0.1.fasta',
    prefix="sample_0.1", 
    model="JC",
)
iqtree_inference.run_iqtree_asr()

total_mut, per_branch_mut = iqtree_inference.count_mutations()
print("n mut in the sampled transmission tree:", total_mut)

In [ ]:
metrics = Metrics()

pd = metrics.tree_distance("./full_tt.nwk", "./sample_0.01.treefile", metric="faith_pd", alignment_length=29903)
print("rf distance: ", pd)

pd = metrics.tree_distance("./full_tt.nwk", "./sample_0.05.treefile", metric="faith_pd", alignment_length=29903)
print("rf distance: ", pd)

pd = metrics.tree_distance("./full_tt.nwk", "./sample_0.1.treefile", metric="faith_pd", alignment_length=29903)
print("rf distance: ", pd)

In [ ]:
# full transmission tree
full_tt_tree = TransmissionTree(
    tt=sim.make_transtree(),
    ref_seq=list(str(SeqIO.read(REF_FASTA, "fasta").seq)),
)
full_tt_tree.build_tree() 
full_tt_tree.construct_newick()
full_tt_tree.save_newick("./full_tt.nwk")

metrics = Metrics()
n_mut = metrics.count_mutations("./full_tt.nwk")

print("n mut in full transmission tree:", n_mut)